PARSING SQL DATABASES

In [8]:
import sqlite3
import os

os.makedirs("data/databases",exist_ok="True")

creating sample db

In [11]:
conn = sqlite3.connect('data/databases/company.db')
cursor = conn.cursor()

creating sample employee detail table

In [12]:
cursor.execute('''CREATE TABLE IF NOT EXISTS employees
(id INTEGER PRIMARY KEY, name TEXT, role TEXT, department TEXT, salary REAL)''')

creating projects table for employees

In [13]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects
(id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget REAL, lead_id INTEGER)''')

inserting sample data

In [14]:
employees = [
    (1, "Aryan Sharma", "Software Engineer", "Development", 75000.0),
    (2, "Priya Patel", "Data Analyst", "Analytics", 68000.0),
    (3, "Rahul Mehta", "Project Manager", "Management", 95000.0),
    (4, "Sneha Iyer", "UI/UX Designer", "Design", 70000.0),
    (5, "Vikram Singh", "DevOps Engineer", "Infrastructure", 80000.0)
]

projects = [
    (1, "NotesPortal", "In Progress", 150000.0, 3),  # Rahul Mehta
    (2, "Smart Survey System", "Completed", 85000.0, 2),  # Priya Patel
    (3, "E-Commerce Platform", "Planning", 200000.0, 1),  # Aryan Sharma
    (4, "Mobile Chat App", "In Progress", 120000.0, 5),  # Vikram Singh
    (5, "Company Dashboard", "Testing", 95000.0, 2)  # Priya Patel
]

In [15]:
cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?,?,?,?,?)',employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?,?,?,?,?)',projects)

In [16]:
conn.commit()
conn.close()

# DATABASE CONTENT EXTRACTION

In [17]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

In [18]:
# METHOD 1 SQLDATABASE UTILITY
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")
print(f"Tables: {db.get_usable_table_names()}")
print(f"Table INFO")
print(db.get_table_info())

Tables: ['employees', 'projects']
Table INFO

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Aryan Sharma	Software Engineer	Development	75000.0
2	Priya Patel	Data Analyst	Analytics	68000.0
3	Rahul Mehta	Project Manager	Management	95000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	NotesPortal	In Progress	150000.0	3
2	Smart Survey System	Completed	85000.0	2
3	E-Commerce Platform	Planning	200000.0	1
*/


In [24]:
# METHOD 2 CUSTOM SQL TO DOCUMENT CONVERTER
from typing import List
from langchain_core.documents import Document

def sql_to_docs(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    documents=[]

    # strat 1 creating docs for each table

    # getting table names
    cursor.execute("SELECT name from sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]

        # getting table schema
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        # getting table data
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()

        # creating table overview doc
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        # adding sample records
        table_content += "Sample Records: \n"
        for row in rows[:5]:
            record = dict(zip(column_names,row))
            table_content += f"{record}\n"

        doc = Document(
            page_content = table_content,
            metadata = {
                'source':db_path,
                'table_name':table_name,
                'num_record':len(rows),
                'data_type':'sql_table'
            }
        )
        documents.append(doc)
    return documents


In [25]:
sql_to_docs("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_record': 5, 'data_type': 'sql_table'}, page_content="Table: employees\nColumns: id, name, role, department, salary\nTotal Records: 5\n\nSample Records: \n{'id': 1, 'name': 'Aryan Sharma', 'role': 'Software Engineer', 'department': 'Development', 'salary': 75000.0}\n{'id': 2, 'name': 'Priya Patel', 'role': 'Data Analyst', 'department': 'Analytics', 'salary': 68000.0}\n{'id': 3, 'name': 'Rahul Mehta', 'role': 'Project Manager', 'department': 'Management', 'salary': 95000.0}\n{'id': 4, 'name': 'Sneha Iyer', 'role': 'UI/UX Designer', 'department': 'Design', 'salary': 70000.0}\n{'id': 5, 'name': 'Vikram Singh', 'role': 'DevOps Engineer', 'department': 'Infrastructure', 'salary': 80000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'projects', 'num_record': 5, 'data_type': 'sql_table'}, page_content="Table: projects\nColumns: id, name, status, budget, lead_id\nTotal Re

In [27]:
# METHOD 2 CUSTOM SQL TO DOCUMENT CONVERTER
from typing import List
from langchain_core.documents import Document

def sql_to_docs2(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    documents=[]

    # strat 2 creating relational documents
    cursor.execute("""SELECT e.name,e.role,p.name as project_name,p.status 
                   FROM employees e 
                   JOIN projects p ON e.id = p.lead_id
""")
    
    relationships = cursor.fetchall()
    rel_content = "Employee Project Relationship: \n\n"
    for rel in relationships:
        rel_content += f"{rel[0]} ({rel[1]}) leads {rel[2]} - Status: {rel[3]}\n"

    rel_doc = Document(
        page_content =  rel_content,
        metadata = {
            'source':db_path,
            'data_type':'sql_relationship',
            'query':'employee_project_join'
        }
    )
    documents.append(rel_doc)
    conn.close()
    return documents


In [28]:
sql_to_docs2("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'data_type': 'sql_relationship', 'query': 'employee_project_join'}, page_content='Employee Project Relationship: \n\nRahul Mehta (Project Manager) leads NotesPortal - Status: In Progress\nPriya Patel (Data Analyst) leads Smart Survey System - Status: Completed\nAryan Sharma (Software Engineer) leads E-Commerce Platform - Status: Planning\nVikram Singh (DevOps Engineer) leads Mobile Chat App - Status: In Progress\nPriya Patel (Data Analyst) leads Company Dashboard - Status: Testing\n')]